In [62]:
import requests
import json
from dotenv import load_dotenv
import os
import pandas as pd
load_dotenv()
API_KEY = os.getenv("API_KEY")
authors_file = open('authors.txt','r')


### Extracting Open Alex by calling the API on names extracted from Web Scraping

In [63]:
import requests
import pandas as pd
import time

rows = []
batch_size = 25 # number of authors per batch
sleep_time = 1   # seconds to wait between batches

authors = [a.strip() for a in authors_file if a.strip()]
total = len(authors)

for start in range(0, total, batch_size):
    batch = authors[start:start + batch_size]
    for author in batch:
        URL = "https://api.openalex.org/authors"
        params = {
            "search": author,
            "select": "display_name,works_count,id,works_api_url,last_known_institutions,summary_stats",
            "api_key": API_KEY
        }
        try:
            response = requests.get(URL, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
        except requests.exceptions.RequestException as e:
            print(f"Request failed for {author}: {e}")
            continue

        if data.get("results"):
            author_data = data["results"][0]
            h_index = author_data.get("summary_stats", {}).get("h_index")
            institutions = author_data.get("last_known_institutions", [])
            country_code = institutions[0].get("country_code") if institutions else None

            rows.append({
                "searched_name": author,
                "display_name": author_data.get("display_name"),
                "works_count": author_data.get("works_count"),
                "author_id": author_data.get("id"),
                "works_api_url": author_data.get("works_api_url"),
                "h_index": h_index,
                "country_code": country_code
            })
        else:
            print(f"No results found for author: {author}")

    print(f"Processed batch {start // batch_size + 1} / {total // batch_size + 1}")
    time.sleep(sleep_time)  # pause between batches to avoid throttling

df = pd.DataFrame(rows)
df.to_csv("authors_data.csv", index=False)

No results found for author: Adam (Zhengzi) Zhou
No results found for author: Amirhossein Nakhaei
Processed batch 1 / 64
No results found for author: Ic2S2 Program Chairs
No results found for author: Vincent Christoph Brockers
Processed batch 2 / 64
No results found for author: Justyna Janczy
Processed batch 3 / 64
Processed batch 4 / 64
Processed batch 5 / 64
Processed batch 6 / 64
Processed batch 7 / 64
No results found for author: Markus Bo Pettersson
No results found for author: Jesica Olivares
Processed batch 8 / 64
Processed batch 9 / 64
No results found for author: Kareem Elrafei
No results found for author: Sandrine Lara Chausson
Processed batch 10 / 64
No results found for author: Dominik Loibner
No results found for author: Alejandro De La Fuente Cuesta
No results found for author: Pamina Syed Al
Processed batch 11 / 64
No results found for author: Adolfo Fuentes-Jofre
No results found for author: Cristian E Candia
Processed batch 12 / 64
No results found for author: Laura Ma